# DEMETER LSTM-AE Rolling-Window Training Notebook 

This notebook trains and evaluates an unsupervised LSTM autoencoder on DEMETER ICE Q3 residual data using rolling time windows.

The workflow is designed for the **weight-updating mode** used in the thesis workflow:

- Window 0 is trained from a freshly initialized model.
- Window `i > 0` is initialized using the best model from window `i-1`.
- The model is trained only on non-seismic sequences.
- Reconstruction-error thresholds are estimated from the training split.
- Validation anomalies are tested for seismic association.
- Storm-contaminated anomaly sequences can be removed using Dst, Kp, and AE thresholds.

The main output is a CSV summary containing the anomaly counts, storm-corrected anomaly counts, seismic-association rate, uncertainty, and matched earthquake count for each rolling window.

> This notebook assumes that the local `lstm` package from the DEMETER project is available and that the preprocessed data files already exist on disk.

## 1. What this notebook requires

Before running this notebook, confirm that the following project files are available:

1. DEMETER residual dataframe  
   `Down_Orbits-Q3-loc-Train-limitedgrids_RES.pkl`

2. Background-corrected rolling-window files  
   `Background_data-window_0.pkl`, `Background_data-window_1.pkl`, ...

3. Earthquake catalogue CSV with at least a `Time` column  
   Example: `EQ.csv`

4. Storm index file  
   `storm_data.pkl`, containing `Datetime`, `Dst`, `Kp`, and `AE`

5. Precomputed seismic label files for train and validation windows  
   Example:
   -
`summary_df_train_30D-22SW-{tag_l}.csv` and `summary_df_val_30D-22SW-{tag_l}.csv`.

6. The local project package `lstm`, containing:
   - `HalfOrbitPairDataset`
   - `scale_datasets`
   - `LSTMAutoencoder`
   - `train_lstm_ae_mode`
   - `AnomalyDetector`
   - `SeismicCriteria`
   - `SeismicAnalysis`

The notebook does **not** regenerate all preprocessing products. It expects the DEMETER background-corrected files and label files to already be present.

## 2. Imports and project path setup

Set `PROJECT_ROOT` to the root folder containing the local DEMETER `lstm` package.

On the DICLUB cluster, this was originally:

```text
/home/mbabu/GRID-METHODS/LSTM-AE-HO/Demeter
```

For another machine, change only the path variables in the configuration cell.

In [19]:
from pathlib import Path
import sys
import os
import glob
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader

# Optional imports retained for compatibility with older project notebooks.
# They are not all used directly in the main rolling-window loop.
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from numpy.lib.stride_tricks import sliding_window_view
from shapely.geometry import Point, Polygon

plt.style.use("default")
sns.set_style("whitegrid")

In [20]:


PROJECT_ROOT = Path("D:\GIT\Demeter-Anomaly-Detection-Framework\Final-Code\lstm")
PROJECT = Path("D:\GIT\Demeter-Anomaly-Detection-Framework")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from lstm import HalfOrbitPairDataset
from lstm.scaling import scale_datasets
from lstm import LSTMAutoencoder
from lstm.anomaly_detection import AnomalyDetector
from lstm import SeismicCriteria
from lstm import train_lstm_ae_mode
from lstm import SeismicAnalysis

## 3. Reproducibility settings

The seed is fixed for NumPy and PyTorch. This reduces run-to-run variability, although exact reproducibility can still depend on GPU/CUDA settings, dataloader behavior, and library versions.

In [21]:
SEED = 201894

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


## 4. Configuration

This is the main cell to edit.

Important parameters:

- `train_months = 12`: each model is trained using 12 months of data.
- `val_months = 3`: each model is validated on the following 3 months.
- `stride = 3`: the rolling window moves forward by 3 months.
- `tw = 48`: earthquake matching time window in hours.
- `sw = 22`: spatial window width in degrees. Change to `22` if running the final thesis setting.
- `pc = 98`: percentile threshold used for reconstruction-error anomaly detection.
- `num_epochs = 1000`: maximum training epochs.
- `patience = 150`: early-stopping patience.
- `m = f"tw{tw}-30dBG_A"`: model/run label. Here `_A` indicates weight-updating mode.

In [22]:
# ============================================================
# DATA AND OUTPUT PATHS
# ============================================================

DATA_DIR = PROJECT / "DATA"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"

STORM_DATA_PATH = DATA_DIR / "storm_data.pkl"
MAIN_DATA_PATH = DATA_DIR / "Down_Orbits-Q3-loc-Train-limitedgrids_RES.pkl"

# Change this if the earthquake catalogue is stored elsewhere.
EQ_PATH = Path(fr"{DATA_DIR}\EQ.csv")
# Alternative example:
# EQ_PATH = DATA_DIR / "Main_earthquakes.csv"

OUTPUT_DIR_RETRAINING = PROJECT_ROOT / "outputs" 
OUTPUT_DIR_RESULTS = PROJECT_ROOT / "outputs" 
OUTPUT_MODEL_DIR = PROJECT_ROOT / "Model-Retraining"

OUTPUT_DIR_RETRAINING.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_RESULTS.mkdir(parents=True, exist_ok=True)
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# ROLLING-WINDOW PARAMETERS
# ============================================================

train_months = 12
val_months = 3
stride = 3
min_data_points = 17

WINDOW_START = "2005-01-01 00:00:00"
WINDOW_END = "2010-01-02 00:00:00"

# ============================================================
# MODEL AND TRAINING PARAMETERS
# ============================================================

input_size = 11
latent_dim = 2
hidden_size = 8
num_layers = 2

batch_size = 8
lr = 0.0001
num_epochs = 10
patience = 150

# ============================================================
# ANOMALY AND SEISMIC-ASSOCIATION PARAMETERS
# ============================================================

pc = 98
tw = 48
sw = 20   # Change to 22 if your thesis/final experiment uses 22° x 22°.

m = f"tw{tw}-30dBG_A-re"

# ============================================================
# STORM FILTER THRESHOLDS
# ============================================================

Dst = -50   # Remove if Dst < -50 nT
Kp = 3      # Remove if Kp > 3
AE = 500    # Remove if AE > 500 nT

print("Run label:", m)
print("Project root:", PROJECT_ROOT)
print("Results directory:", OUTPUT_DIR_RESULTS)
print("Model directory:", OUTPUT_MODEL_DIR)

Run label: tw48-30dBG_A-re
Project root: D:\GIT\Demeter-Anomaly-Detection-Framework\Final-Code\lstm
Results directory: D:\GIT\Demeter-Anomaly-Detection-Framework\Final-Code\lstm\outputs
Model directory: D:\GIT\Demeter-Anomaly-Detection-Framework\Final-Code\lstm\Model-Retraining


## 5. Load DEMETER data, earthquake catalogue, and storm data

The main DEMETER dataframe contains residual features and location/time information.

Columns beginning with `Q3` are removed here because the model uses the already prepared residual/feature columns rather than raw Q3 columns.

In [23]:
storm_data = pd.read_pickle(STORM_DATA_PATH)
data = pd.read_pickle(MAIN_DATA_PATH)

# Drop raw Q3 columns that are not required for the model input.
df = data.loc[:, ~data.columns.str.startswith("Q3")].copy()
df.index = pd.to_datetime(df.index)

eq = pd.read_csv(EQ_PATH, parse_dates=["Time"])
eq['Time'] = pd.to_datetime(eq['Time'], errors="coerce")
eq['Time'] = pd.to_datetime(eq['Time']).dt.strftime('%Y-%m-%d %H:%M:%S')
eq['Time'] = pd.to_datetime(eq['Time'])


storm_data["Datetime"] = pd.to_datetime(storm_data["Datetime"], errors="coerce")

storm_all = storm_data[
    (storm_data["Dst"] < Dst) |
    (storm_data["Kp"] > Kp) |
    (storm_data["AE"] > AE)
].dropna(subset=["Datetime"]).copy()

print("DEMETER dataframe shape:", df.shape)
print("Earthquake catalogue shape:", eq.shape)
print("Storm-filtered rows:", storm_all.shape)

DEMETER dataframe shape: (456539, 13)
Earthquake catalogue shape: (12442, 5)
Storm-filtered rows: (6318, 4)


## 6.anomaly detection for a split

This function computes reconstruction errors and applies percentile thresholds.

Threshold logic:

- For training data, thresholds are estimated from the training reconstruction-error distribution.
- For validation data, the training threshold is reused.
- This avoids defining a new threshold from validation data.

In [24]:
def process_split(
    name,
    model,
    dataloader,
    num_features,
    threshold_agg=None,
    threshold_fb=None,
    threshold_percentile=None,
    plot=False,
):
    detector = AnomalyDetector(model, dataloader=dataloader, num_features=num_features)

    errors_agg = detector.compute_reconstruction_errors_agg(dataloader)
    errors_fb = detector.compute_reconstruction_errors_fb(dataloader)

    if threshold_agg is None:
        threshold_agg = np.percentile(errors_agg, threshold_percentile)

    if threshold_fb is None:
        threshold_fb = np.percentile(errors_fb, threshold_percentile, axis=0)

    anomalies_agg = detector.detect_anomalies_agg(errors_agg, threshold_agg)
    anomalies_fb = detector.detect_anomalies_fb(errors_fb, threshold_fb)

    if plot:
        detector.plot_feature_errors(
            errors_fb,
            title=f"{name} Data Error Distribution",
            threshold=threshold_fb,
            percentile=threshold_percentile,
            save_path=str(OUTPUT_DIR_RESULTS / f"FB_Error-{name}_model{m}.png"),
        )

        detector.plot_reconstruction_error(
            errors_agg,
            title=f"{name} Data Error Distribution",
            threshold=threshold_agg,
            percentile=threshold_percentile,
            save_path=str(OUTPUT_DIR_RESULTS / f"Agg_Error-{name}_model{m}.png"),
        )

    return {
        "detector": detector,
        "errors_agg": errors_agg,
        "errors_fb": errors_fb,
        "threshold_agg": threshold_agg,
        "threshold_fb": threshold_fb,
        "anomalies_agg": anomalies_agg,
        "anomalies_fb": anomalies_fb,
    }

## 7.  storm correction

This function removes anomaly sequences that overlap with geomagnetic storm intervals.

A sequence is marked storm-affected when at least one storm timestamp falls between the start and end time of that sequence.

In [25]:
def correct_anomalies_for_storms(datetime_sequences, anomalies_agg, anomalies_fb, storm_all):
    storm_data_indices = []
    storm_times = pd.to_datetime(storm_all["Datetime"], errors="coerce").dropna()

    for idx, seq in enumerate(datetime_sequences):
        seq_times = pd.to_datetime(seq)
        start_t, end_t = seq_times[0], seq_times[-1]

        if storm_times.between(start_t, end_t).any():
            storm_data_indices.append(idx)

    corrected_anomalies_agg = [
        idx for idx in anomalies_agg
        if idx not in storm_data_indices
    ]

    corrected_anomalies_fb = [[] for _ in range(len(anomalies_fb))]

    for feature_index, anomalies in enumerate(anomalies_fb):
        corrected_anomalies_fb[feature_index] = [
            idx for idx in anomalies
            if idx not in storm_data_indices
        ]

    return corrected_anomalies_agg, corrected_anomalies_fb, storm_data_indices

## 8. seismic association analysis

This function uses the existing `SeismicAnalysis` class from the local project package.

For a given list of anomaly indices, it returns:

- seismic-association percentage,
- binomial uncertainty,
- number of matched earthquakes,
- seismic anomaly indices.

The uncertainty is estimated using a binomial proportion error. Edge cases are handled when all or none of the anomaly sequences are seismic-associated.

In [26]:
def run_seismic_analysis(
    dataset,
    test_dataset,
    eq,
    seismic_criteria,
    corrected_test_fb_anomalies,
    corrected_test_anomalies,
    model_name,
    output_dir,
    data_label="",
):
    sa = SeismicAnalysis(
        dataset=dataset,
        earthquake_catalog=eq,
        create_half_orbit_sequences=test_dataset.create_half_orbit_sequences,
        is_eq_fn=seismic_criteria.is_eq,
        model_name=f"{model_name}",
        threshold_label="98",
        mean_rst=0,
        sigma_rst=0,
        output_dir=output_dir,
    )

    seismic_seqs_agg, matched_eqs_agg, missed_eqs_agg = sa.agg_analysis(
        data_label=data_label,
        anomalous_indices=corrected_test_anomalies,
        plot=False,
    )

    total_sequences_agg = len(corrected_test_anomalies)
    seismic_count = len(seismic_seqs_agg)
    total_eq = len(matched_eqs_agg)

    if total_sequences_agg > 0:
        if seismic_count == total_sequences_agg:
            p_agg = (seismic_count - 1) / total_sequences_agg
        elif seismic_count == 0:
            p_agg = 1 / total_sequences_agg
        else:
            p_agg = seismic_count / total_sequences_agg

        agg_value = p_agg * 100
        agg_error = np.sqrt(p_agg * (1 - p_agg) / total_sequences_agg) * 100
    else:
        agg_value, agg_error = 0, 0

    return {
        "agg": {
            "value": agg_value,
            "error": agg_error,
            "total_eq": total_eq,
            "seismic_indices": seismic_seqs_agg,
        }
    }

## 9.  rolling-window generation

Each window has:

- 12 months for training,
- immediately followed by 3 months for validation,
- then shifted by 3 months.

This gives non-overlapping validation windows and overlapping training windows.

In [27]:
def generate_windows(df, start_date, end_date, train_months, val_months):
    windows = []
    current = pd.to_datetime(start_date)
    final = pd.to_datetime(end_date)

    while current + pd.DateOffset(months=train_months + val_months) < final:
        train_start = current
        train_end = current + pd.DateOffset(months=train_months)
        val_end = train_end + pd.DateOffset(months=val_months)

        windows.append({
            "train_start": train_start,
            "train_end": train_end,
            "val_start": train_end,
            "val_end": val_end,
        })

        current = current + pd.DateOffset(months=stride)

    return windows

## 10. reconstruction-error distribution plot

This plot compares the log reconstruction-error distributions for the train and validation splits.

The vertical dashed lines mark the chosen percentile thresholds.

In [28]:
def plot_error_distributions(i, m, train, val, pc=98, output_dir=None, save=False):
    errors_train = pd.Series(train["errors_agg"])
    errors_val = pd.Series(val["errors_agg"])

    threshold_train = np.percentile(np.log(errors_train), pc)
    threshold_val = np.percentile(np.log(errors_val), pc)

    datasets = [
        (f"Train_w{i}", np.log(errors_train), threshold_train, "tab:blue"),
        (f"Validation_w{i}", np.log(errors_val), threshold_val, "tab:orange"),
    ]

    plt.figure(figsize=(13, 7))
    sns.set_style("whitegrid")

    for label, errors, thr, color in datasets:
        sns.kdeplot(errors, fill=True, alpha=0.20, color=color, label=label)
        plt.axvline(thr, color=color, ls="--", lw=1.8)
        plt.text(
            thr,
            plt.ylim()[1] * 0.80,
            f"{thr:.2f}",
            rotation=90,
            color=color,
            va="bottom",
            ha="right",
            fontsize=10,
        )

    plt.xlabel("Error (log)")
    plt.ylabel("Density")
    plt.title(f"Reconstruction error distributions with {pc}th percentile thresholds")
    plt.legend(title="Datasets", loc="upper right")
    plt.tight_layout()

    if save and output_dir is not None:
        fname = Path(output_dir) / f"Aggregate_Errordistribution-pc{pc}-HP_m{m}_w{i}.png"
        plt.savefig(fname, dpi=300, bbox_inches="tight")
        plt.close()
    else:
        plt.show()

## 11. Generate the rolling windows

Check the printed windows before starting the long training loop.

In [29]:
windows_train = generate_windows(
    df,
    start_date=WINDOW_START,
    end_date=WINDOW_END,
    train_months=train_months,
    val_months=val_months,
)

print("Number of rolling windows:", len(windows_train))

for i, w in enumerate(windows_train):
    print(
        f"Window {i}: "
        f"Train {w['train_start']} -> {w['train_end']} | "
        f"Val {w['val_start']} -> {w['val_end']}"
    )

Number of rolling windows: 16
Window 0: Train 2005-01-01 00:00:00 -> 2006-01-01 00:00:00 | Val 2006-01-01 00:00:00 -> 2006-04-01 00:00:00
Window 1: Train 2005-04-01 00:00:00 -> 2006-04-01 00:00:00 | Val 2006-04-01 00:00:00 -> 2006-07-01 00:00:00
Window 2: Train 2005-07-01 00:00:00 -> 2006-07-01 00:00:00 | Val 2006-07-01 00:00:00 -> 2006-10-01 00:00:00
Window 3: Train 2005-10-01 00:00:00 -> 2006-10-01 00:00:00 | Val 2006-10-01 00:00:00 -> 2007-01-01 00:00:00
Window 4: Train 2006-01-01 00:00:00 -> 2007-01-01 00:00:00 | Val 2007-01-01 00:00:00 -> 2007-04-01 00:00:00
Window 5: Train 2006-04-01 00:00:00 -> 2007-04-01 00:00:00 | Val 2007-04-01 00:00:00 -> 2007-07-01 00:00:00
Window 6: Train 2006-07-01 00:00:00 -> 2007-07-01 00:00:00 | Val 2007-07-01 00:00:00 -> 2007-10-01 00:00:00
Window 7: Train 2006-10-01 00:00:00 -> 2007-10-01 00:00:00 | Val 2007-10-01 00:00:00 -> 2008-01-01 00:00:00
Window 8: Train 2007-01-01 00:00:00 -> 2008-01-01 00:00:00 | Val 2008-01-01 00:00:00 -> 2008-04-01 00:00:0

## 12. Main rolling-window training and evaluation loop

This is the main execution cell.

For each rolling window, the notebook performs the following steps:

1. Load the corresponding background-corrected window file.
2. Split the data into train and validation periods.
3. Build half-orbit pair datasets.
4. Load precomputed seismic labels.
5. Keep only non-seismic sequences for model training.
6. Scale the data using the training-normal subset.
7. Train the LSTM autoencoder using `train_lstm_ae_mode`.
8. For window `i > 0`, warm-start from the previous window's best model.
9. Estimate the training reconstruction-error threshold.
10. Apply the same threshold to validation data.
11. Remove storm-contaminated anomaly sequences.
12. Compute seismic-association rates before and after storm correction.
13. Append the result to a summary dataframe.

The model is trained using your project function:

```python
train_lstm_ae_mode(...)
```

That function uses Adam optimization, cosine-annealing learning-rate scheduling, early stopping, and optional epoch checkpoint saving.

In [33]:
all_results = []

for i, w in enumerate(windows_train):
    tag_l = f"tw{tw}_w{i}"
    tag = f"Hp_{m}_w{i}"

    print("\n" + "=" * 90)
    print(
        f"Window {i}: Train {w['train_start']} -> {w['train_end']} | "
        f"Val {w['val_start']} -> {w['val_end']}"
    )
    print("=" * 90)

    # ------------------------------------------------------------
    # Load background-corrected data for this rolling window
    # ------------------------------------------------------------
    bg_window_path = BG_WINDOW_DIR / f"Background_data-window_{i}.pkl"

    if not bg_window_path.exists():
        raise FileNotFoundError(f"Missing background window file: {bg_window_path}")

    dfw = pd.read_pickle(bg_window_path)
    dfw = dfw.loc[:, ~dfw.columns.str.startswith("Q3")].copy()
    dfw.index = pd.to_datetime(dfw.index)

    train_set = dfw[(dfw.index >= w["train_start"]) & (dfw.index < w["train_end"])]
    val_set = dfw[(dfw.index >= w["val_start"]) & (dfw.index < w["val_end"])]

    # Test set is kept for compatibility with scale_datasets API.
    test_set = pd.DataFrame(df[df.index >= "2010-01-01"])

    # ------------------------------------------------------------
    # Build datasets
    # ------------------------------------------------------------
    train_dataset = HalfOrbitPairDataset(train_set, min_data_points=min_data_points)
    val_dataset = HalfOrbitPairDataset(val_set, min_data_points=min_data_points)
    test_dataset = HalfOrbitPairDataset(test_set, min_data_points=min_data_points)

    seismic_criteria = SeismicCriteria(spatial_width=sw, time_window_hours=tw)

    # ------------------------------------------------------------
    # Load seismic labels for normal-only training
    # ------------------------------------------------------------
    train_label_path = Path(
        fr"{DATA_DIR}\Label_data\summary_df_train_30D-22SW-{tag_l}.csv"
    )

    val_label_path = Path(
        fr"{DATA_DIR}\Label_data\summary_df_val_30D-22SW-{tag_l}.csv"
    )

    if not train_label_path.exists():
        raise FileNotFoundError(f"Missing train label file: {train_label_path}")

    if not val_label_path.exists():
        raise FileNotFoundError(f"Missing validation label file: {val_label_path}")

    df_train_labels = pd.read_csv(train_label_path)
    df_val_labels = pd.read_csv(val_label_path)

    train_dataset_normal = HalfOrbitPairDataset(
        train_set,
        df_train_labels,
        min_data_points,
        use_label_0_only=True,
    )

    val_dataset_normal = HalfOrbitPairDataset(
        val_set,
        df_val_labels,
        min_data_points,
        use_label_0_only=True,
    )

    # ------------------------------------------------------------
    # Scale normal training and validation data
    # ------------------------------------------------------------
    scaled_train_data_n, scaled_val_data_n, scaled_test_data, mean = scale_datasets(
        train_dataset_normal,
        train_dataset,
        val_dataset,
        test_dataset,
        fit=True,
    )

    train_data_loader_n = DataLoader(
        scaled_train_data_n,
        batch_size=batch_size,
        shuffle=True,
    )

    val_data_loader_n = DataLoader(
        scaled_val_data_n,
        batch_size=batch_size,
        shuffle=False,
    )

    print("TRAINING DATA INFORMATION")
    print("length of train_loader:", len(train_data_loader_n))
    print("length of val_loader:", len(val_data_loader_n))
    print("length of train_data:", len(scaled_train_data_n))
    print("length of val_data:", len(scaled_val_data_n))

    # ------------------------------------------------------------
    # Build model
    # ------------------------------------------------------------
    model = LSTMAutoencoder(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        latent_dim=latent_dim,
    )

    all_epochs_dir = OUTPUT_MODEL_DIR / f"check_{tag}"
    best_model_path = OUTPUT_MODEL_DIR / f"best_model_{tag}.pth"
    loss_plot_path = OUTPUT_MODEL_DIR / f"loss_curve_model_{tag}.png"
    loss_csv_path = OUTPUT_MODEL_DIR / f"loss_history_{tag}.csv"

    all_epochs_dir.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------
    # Weight-updating mode
    # Window 0: fresh initialization
    # Window i>0: initialize from previous best model
    # ------------------------------------------------------------
    if i > 0:
        prev_tag = f"Hp_{m}_w{i-1}"
        prev_best = OUTPUT_MODEL_DIR / f"best_model_{prev_tag}.pth"

        try:
            model.load_state_dict(torch.load(prev_best, map_location=torch.device("cpu")))
            print(f"[Warm start] Loaded previous best model: {prev_best}")
        except Exception as e:
            print(f"[Warm start skipped] {e}")

    # ------------------------------------------------------------
    # Train 
    # ------------------------------------------------------------
    model, train_losses, val_losses = train_lstm_ae_mode(
        model,
        train_data_loader_n,
        val_data_loader_n,
        num_epochs=num_epochs,
        lr=lr,
        patience=patience,
        early_stopping=True,
        save_all_epochs=True,
        all_epochs_dir=str(all_epochs_dir),
        best_model_path=str(best_model_path),
        plot_loss=True,
        loss_plot_path=str(loss_plot_path),
        input_size=input_size,
        hidden_size=hidden_size,
        latent_dim=latent_dim,
        num_layers=num_layers,
        batch_size=batch_size,
        seq=17,
        mode="autoencoder",
    )

    # Save loss history as a CSV for later inspection.
    loss_df = pd.DataFrame({
        "epoch": np.arange(1, len(train_losses) + 1),
        "train_loss": train_losses,
        "val_loss": val_losses,
    })
    loss_df.to_csv(loss_csv_path, index=False)

    print(f"[OK] Saved best model -> {best_model_path}")
    print(f"[OK] Saved loss curve -> {loss_plot_path}")
    print(f"[OK] Saved loss history -> {loss_csv_path}")

    # ------------------------------------------------------------
    # Recreate scaled data for evaluation on unfiltered train and validation sets
    # ------------------------------------------------------------
    scaled_train_data, scaled_val_data, _, _ = scale_datasets(
        train_dataset_normal,
        train_dataset,
        val_dataset,
        test_dataset,
        fit=False,
    )

    train_data_loader = DataLoader(
        scaled_train_data,
        batch_size=1,
        shuffle=False,
    )

    val_data_loader = DataLoader(
        scaled_val_data,
        batch_size=1,
        shuffle=False,
    )

    print("RESULTS DATA INFORMATION")
    print("length of train_loader:", len(train_data_loader))
    print("length of val_loader:", len(val_data_loader))
    print("length of train_data:", len(scaled_train_data))
    print("length of val_data:", len(scaled_val_data))

    # ------------------------------------------------------------
    # Anomaly detection
    # ------------------------------------------------------------
    train = process_split(
        f"Train_{tag_l}",
        model,
        train_data_loader,
        threshold_percentile=pc,
        num_features=input_size,
        plot=False,
    )

    val = process_split(
        f"Validation_{tag_l}",
        model,
        val_data_loader,
        num_features=input_size,
        threshold_agg=train["threshold_agg"],
        threshold_fb=train["threshold_fb"],
        plot=False,
    )

    plot_error_distributions(
        i,
        m,
        train,
        val,
        pc=pc,
        output_dir=OUTPUT_DIR_RESULTS,
        save=True,
    )

    # ------------------------------------------------------------
    # Storm correction
    # ------------------------------------------------------------
    val_sequences, val_datetime_sequences, val_lat_long_sequences, _ = (
        val_dataset.create_half_orbit_sequences(val_set)
    )

    train_sequences, train_datetime_sequences, train_lat_long_sequences, _ = (
        train_dataset.create_half_orbit_sequences(train_set)
    )

    corrected_val_anomalies_agg, corrected_val_anomalies_fb, storm_val_data_indices = (
        correct_anomalies_for_storms(
            val_datetime_sequences,
            val["anomalies_agg"],
            val["anomalies_fb"],
            storm_all,
        )
    )

    corrected_train_anomalies_agg, corrected_train_anomalies_fb, storm_train_data_indices = (
        correct_anomalies_for_storms(
            train_datetime_sequences,
            train["anomalies_agg"],
            train["anomalies_fb"],
            storm_all,
        )
    )

    val_anomalies_before = len(val["anomalies_agg"])
    val_anomalies_after = len(corrected_val_anomalies_agg)

    train_anomalies_before = len(train["anomalies_agg"])
    train_anomalies_after = len(corrected_train_anomalies_agg)

    # ------------------------------------------------------------
    # Seismic association analysis
    # ------------------------------------------------------------
    results_val_sc = run_seismic_analysis(
        val_set,
        val_dataset,
        eq,
        seismic_criteria,
        corrected_val_anomalies_fb,
        corrected_val_anomalies_agg,
        model_name=f"stormC_EQ_{tag}",
        output_dir=str(OUTPUT_DIR_RESULTS),
        data_label=f"Val_{tag_l}",
    )

    results_train_sc = run_seismic_analysis(
        train_set,
        train_dataset,
        eq,
        seismic_criteria,
        corrected_train_anomalies_fb,
        corrected_train_anomalies_agg,
        model_name=f"stormC_EQ_{tag}",
        output_dir=str(OUTPUT_DIR_RESULTS),
        data_label=f"Train_{tag_l}",
    )

    results_val = run_seismic_analysis(
        val_set,
        val_dataset,
        eq,
        seismic_criteria,
        val["anomalies_fb"],
        val["anomalies_agg"],
        model_name=f"EQ_{tag}",
        output_dir=str(OUTPUT_DIR_RESULTS),
        data_label=f"Val_{tag_l}",
    )

    results_train = run_seismic_analysis(
        train_set,
        train_dataset,
        eq,
        seismic_criteria,
        train["anomalies_fb"],
        train["anomalies_agg"],
        model_name=f"EQ_{tag}",
        output_dir=str(OUTPUT_DIR_RESULTS),
        data_label=f"Train_{tag_l}",
    )

    # ------------------------------------------------------------
    # Store window results
    # ------------------------------------------------------------
    all_results.append({
        "window": i,
        "split": "Val",
        "train_start": w["train_start"],
        "train_end": w["train_end"],
        "val_start": w["val_start"],
        "val_end": w["val_end"],
        "anomalies": val_anomalies_before,
        "anomalies_sc": val_anomalies_after,
        "agg_value": results_val["agg"]["value"],
        "agg_error": results_val["agg"]["error"],
        "agg_value_sc": results_val_sc["agg"]["value"],
        "agg_error_sc": results_val_sc["agg"]["error"],
        "agg_total_eq": results_val["agg"]["total_eq"],
        "agg_total_eq_sc": results_val_sc["agg"]["total_eq"],
        "seismic_indices": results_val["agg"]["seismic_indices"],
        "seismic_indices_sc": results_val_sc["agg"]["seismic_indices"],
        "threshold_agg": train["threshold_agg"],

    })

    all_results.append({
        "window": i,
        "split": "Train",
        "train_start": w["train_start"],
        "train_end": w["train_end"],
        "val_start": w["val_start"],
        "val_end": w["val_end"],
        "anomalies": train_anomalies_before,
        "anomalies_sc": train_anomalies_after,
        "agg_value": results_train["agg"]["value"],
        "agg_error": results_train["agg"]["error"],
        "agg_value_sc": results_train_sc["agg"]["value"],
        "agg_error_sc": results_train_sc["agg"]["error"],
        "agg_total_eq": results_train["agg"]["total_eq"],
        "agg_total_eq_sc": results_train_sc["agg"]["total_eq"],
        "seismic_indices": results_train["agg"]["seismic_indices"],
        "seismic_indices_sc": results_train_sc["agg"]["seismic_indices"],
        "threshold_agg": train["threshold_agg"],

    })

    interim_df = pd.DataFrame(all_results)
    interim_csv = OUTPUT_DIR_RESULTS / f"result_Hp_{m}_INTERIM.csv"
    interim_df.to_csv(interim_csv, index=False)

    print(interim_df.tail(2))
    print(f"[OK] Saved interim result CSV -> {interim_csv}")


Window 0: Train 2005-01-01 00:00:00 -> 2006-01-01 00:00:00 | Val 2006-01-01 00:00:00 -> 2006-04-01 00:00:00
TRAINING DATA INFORMATION
length of train_loader: 93
length of val_loader: 48
length of train_data: 742
length of val_data: 381
Epoch [1/10], Train Loss: 1.0305, Val Loss: 1.2163
Epoch [2/10], Train Loss: 1.0244, Val Loss: 1.2103
Epoch [3/10], Train Loss: 1.0190, Val Loss: 1.2047
Epoch [4/10], Train Loss: 1.0139, Val Loss: 1.1990
Epoch [5/10], Train Loss: 1.0092, Val Loss: 1.1938
Epoch [6/10], Train Loss: 1.0042, Val Loss: 1.1874
Epoch [7/10], Train Loss: 0.9968, Val Loss: 1.1774


KeyboardInterrupt: 

## 13. Save the final rolling-window result table

The final CSV contains one row for validation and one row for training for each rolling window.

In [ ]:
df_results = pd.DataFrame(all_results)

csv_path = OUTPUT_DIR_RESULTS / f"result_Hp_{m}.csv"
df_results.to_csv(csv_path, index=False)

print(f"Results saved to {csv_path}")
display(df_results)

## 14. Expected outputs

After successful execution, the notebook should produce:

### Model files

Stored in:

```text
models/Model-Retraining/
```

Expected examples:

```text
best_model_Hp_tw48-30dBG_A_w0.pth
best_model_Hp_tw48-30dBG_A_w1.pth
check_Hp_tw48-30dBG_A_w0/
loss_curve_model_Hp_tw48-30dBG_A_w0.png
loss_history_Hp_tw48-30dBG_A_w0.csv
```

### Result files

Stored in:

```text
outputs/New_BG/
```

Expected examples:

```text
result_Hp_tw48-30dBG_A.csv
result_Hp_tw48-30dBG_A_INTERIM.csv
Aggregate_Errordistribution-pc98-HP_mtw48-30dBG_A_w0.png
```

### Main columns in the final CSV

- `window`: rolling-window index
- `split`: Train or Val
- `anomalies`: number of aggregate anomalies before storm correction
- `anomalies_sc`: number of aggregate anomalies after storm correction
- `agg_value`: seismic-association percentage before storm correction
- `agg_error`: binomial uncertainty before storm correction
- `agg_value_sc`: seismic-association percentage after storm correction
- `agg_error_sc`: binomial uncertainty after storm correction
- `agg_total_eq`: number of matched earthquakes before storm correction
- `agg_total_eq_sc`: number of matched earthquakes after storm correction
- `seismic_indices`: anomaly sequence indices satisfying the seismic criterion
- `seismic_indices_sc`: storm-corrected anomaly sequence indices satisfying the seismic criterion

## 15. Notes for users running the notebook

### If the notebook stops because a file is missing

Check that the paths in the configuration cell match the local machine or cluster directory structure.

Most common missing files are:

- background-corrected rolling-window pickle files,
- precomputed seismic label CSV files,
- earthquake catalogue path,
- `storm_data.pkl`.


### If you want the reinitialisation mode

This notebook is written for the weight-updating `_A` mode.

For independent reinitialisation (`In_A` mode) in each window, remove or comment out the block that loads:

```python
prev_best = OUTPUT_MODEL_DIR / f"best_model_{prev_tag}.pth"
model.load_state_dict(...)
```

Then each rolling window will start from a freshly initialized LSTM autoencoder.